# WS3 — The Centerpiece: A Gradient-Boosted Ablation of the Five State Views

**Workstream 3 of the pitch-sequencing rigor ladder — the centerpiece.** This notebook fits and
reads the study's central measurement instrument: gradient-boosted decision trees (LightGBM) run
across **all five** nested state views, for both a next-pitch **behavior** model and a decomposed
event-tree **outcome** model. It is the first workstream that can compute the headline ablation
like-for-like across the whole ladder, the first to test *outcome* (not just selection)
sequencing value, and the supplier of the artifacts every prescriptive workstream consumes.

## The central question (SPEC §6, verbatim)

> Every suitable model is trained on the **same five state variants**. The differences between
> them *are* the sequencing evidence.
>
> $$\Delta_{\text{order}} = \text{Loss}(\min[U, L1]) - \text{Loss}(O), \qquad
>   \Delta_{\text{matchup}} = \text{Loss}(O) - \text{Loss}(OM)$$
>
> Interpretation: O beating C but **not** U/L1 ⇒ history matters but *order* barely does. O
> beating both U and L1 ⇒ genuine ordered dependence.

WS3 answers this on **three** targets at once — next-pitch **selection**, pitch **outcome**, and
**run value** — with pitcher-game clustered confidence intervals, on validation (2024) and the
**locked test** (2025).

## Why THIS workstream is the measurement instrument

Three properties, none shared by the workstreams before it:

1. **It runs all five views through one strong model family.** WS1's tables declared `OM`
   infeasible; WS2's grammar declared `U` and `OM` not-applicable. WS3 fits `C / U / L1 / O / OM`
   with the same gradient-boosted machinery, so `Δ_order` and `Δ_matchup` are finally computed
   like-for-like across the whole ladder (decision D34).
2. **It measures outcomes, not just selection.** It fits the decomposed event-tree outcome stack
   (decision D32) that WS1 could only approximate and WS2 deliberately did not attempt, so it
   carries the project's central number — the outcome-target `Δ_order` (finding #2).
3. **It supplies the prescriptive phase.** WS4 (bandit), WS5 (MDP), and WS7 (offline RL) import
   WS3's saved counterfactual q̂ grid and behavior propensities and never refit their own outcome
   model (decision D33). SPEC §12.3: *"If O doesn't beat U/L1 here, be skeptical of everything
   fancier."*

## The three findings this notebook keeps separate (SPEC §0, verbatim)

> 1. **Selection structure** — prior pitches help predict *what is thrown next*.
> 2. **Predictive sequencing value** — prior pitches help predict the *outcome* of the current
>    pitch, after conditioning on the current pitch and game state.
> 3. **Prescriptive/causal value** — *changing* the sequence would improve outcomes.

WS3 lives in **finding #2**: a boosted ablation is *predictive*, not causal. The q̂ grid it
publishes is `E[R | s, A=f]` (a conditional expectation), **not** `E[R | s, do(A=f)]` (a causal
effect). Turning finding #2 into finding #3 is the OPE/RL workstreams' job, not WS3's — the
firewall we restate throughout.

## The DATA_MODE toggle

This notebook is a **scaffold**. Phase 2 runs it on the real Statcast decision table; here a
single toggle, `DATA_MODE`, selects the world:

- `'synth_positive'` — the oracle's **positive world** (a planted ordered *outcome* effect: a
  velocity-transition whiff boost). **Default**, because it exercises the full outcome ablation
  with a real signal to detect and the negative-`Δ_matchup` cost to display.
- `'synth_null'` — the oracle's **null world** (an ordered *selection* habit, but no ordered
  *outcome* effect). Shows the D20 finding-#1/finding-#2 separation live.
- `'real'` — the real decision table built by `python -m pitchseq.build_table` (Phase 2).

## How to read this notebook

Every code step is bracketed by plain-worded markdown: **before** each cell we say what will
happen and why; **after** each cell we say how to read what came out. Numbers that depend on the
real data are `{PLACEHOLDER}` in the companion `PAPER.md`; here they simply appear when you run
the cell. The **Results** section (§9) is *branched* on two axes (order H1/H2/H3 × matchup
M+/M0/M−), exactly the SPEC §6 grid: a code cell inspects the computed numbers and prints which
branch applies, and the markdown that follows holds the pre-written interpretation for every
branch. The exact formulas live in `THEORY.md`; the exact code in
`workstreams/ws3_gbdt_stack/model.py`; the plain-English tour in `SEAN-README.md`.

> **Scale note.** WS3 is the heaviest CPU step in the study. On the full data a single view takes
> tens of minutes; the synthetic demo below takes a few minutes at the default `N_GAMES`. The
> scale knobs (`N_GAMES`, `NB_PARAMS`, the falsification budgets) are all collected in the setup
> cell so a quick pass is one edit away — lower them for a fast smoke test, raise them for the
> committed-scale demo. Phase-2 real runs go through the CLI (`run_ws3.py`, RUNBOOK WS3.1–3.3)
> with `--tune` and the predeclared defaults, not this notebook's demo params.

## 1. Setup

We put the repository root on `sys.path` (so the `pitchseq` package and the
`workstreams.ws3_gbdt_stack` module import whether the notebook is launched from the repo root or
from `notebooks/`), load the shared study config, and set the knobs that steer the whole notebook.
`DATA_MODE` chooses the world; `VIEWS` is all five (WS3 is the first workstream that runs every
one); `NB_PARAMS` is a light demo LightGBM setting (Phase-2 real runs use `--tune` via the CLI);
the `FALS_*` knobs size the synthetic falsification battery in §8. Nothing here touches data yet.

In [ ]:
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# --- locate the repository root (works from repo root or from notebooks/) ---
REPO_ROOT = Path.cwd()
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "pyproject.toml").exists() and (_p / "workstreams").is_dir():
        REPO_ROOT = _p
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# --- shared foundation (WS0) ---
from pitchseq.config import load_config
from pitchseq.splits import make_splits
from pitchseq.families import FAMILIES, feasible_action_mask
from pitchseq.outcomes import OUTCOME1, OUTCOME2
from pitchseq.states import build_view, STATE_VIEWS
from pitchseq.decision_table import CONTEXT_COLS
from pitchseq.eval.predictions import ACTION_PROB_COLS, OUTCOME1_PROB_COLS, OUTCOME2_PROB_COLS
from pitchseq.eval.metrics import reliability_table, clustered_ci, run_value_calibration
from pitchseq.eval.harness import evaluate_predictions, compare_views, slice_masks

# --- WS3 (this workstream) ---
from workstreams.ws3_gbdt_stack import model as ws3
from workstreams.ws3_gbdt_stack.model import (
    BehaviorModel, OutcomeStack, DEFAULT_PARAMS, HP_GRID, DISAGREEMENT_FLAG_ABS, ACTION_COL,
)
# _runvalue_compare is the exact run-value-MAE ablation the CLI uses; imported for the central table.
from workstreams.ws3_gbdt_stack.run_ws3 import run_ws3, _runvalue_compare

CONFIG = load_config()
SEED = int(CONFIG.get("seeds", {}).get("global", 20260713))

# The world this run analyses: 'synth_positive' | 'synth_null' | 'real'.
# Default 'synth_positive': it plants a real ordered OUTCOME effect (velocity-transition whiff
# boost) so the outcome ablation has a signal to detect and the M- matchup cost to display.
DATA_MODE = "synth_positive"

# WS3 is the first workstream that runs ALL FIVE views (decision D34).
VIEWS = ["C", "U", "L1", "O", "OM"]

# Synthetic-world size for the main pipeline (ignored when DATA_MODE == 'real').
N_GAMES = 120

# Cluster-bootstrap replicates for the CIs (SPEC §7 clusters by pitcher-game). CI width only.
N_BOOT = 100

# Light demo LightGBM params for the in-notebook fits. Phase-2 real runs use the predeclared
# DEFAULT_PARAMS (300 estimators) + the --tune 12-combo grid via the CLI (decision D34); a
# scaffold keeps trees small so all five views fit in a few minutes.
NB_PARAMS = {"n_estimators": 200, "num_leaves": 31, "min_child_samples": 30,
             "learning_rate": 0.05}

# Synthetic falsification battery budgets (§8). These match the committed-validation scale so
# the live D35 verdicts reproduce (NULL_QUIET / MECHANISM_RECOVERED); the battery refits all
# five views plus permutations, so it is ~11 min PER world here (the study's heaviest cell).
# Lower them for a quick smoke test — the verdicts sharpen with scale, so a tiny run may miss.
FALS_N_GAMES = 300
FALS_N_PERM = 24
FALS_CI_BOOT = 60

# Where the real decision table lives after `python -m pitchseq.build_table` (Phase 2).
REAL_TABLE_PATH = REPO_ROOT / "data" / "processed" / "decision_table.parquet"

print(f"repo root : {REPO_ROOT}")
print(f"DATA_MODE : {DATA_MODE}")
print(f"views     : {VIEWS}   (WS3 is the first workstream to run all five, D34)")
print(f"N_GAMES   : {N_GAMES}   NB_PARAMS n_estimators={NB_PARAMS['n_estimators']}")
print(f"seed      : {SEED}")
print(f"lightgbm  : available (WS3's core dependency, the '.[ml]' extra)")

### Plotting style (fixed, colorblind-safe view colours)

Every figure uses the **Okabe–Ito** palette (safe for the common forms of colour-vision
deficiency) and a **fixed view → colour map**, identical to WS1/WS2, so a given state view is
*always* the same colour throughout the study: **C** blue, **U** orange, **L1** bluish-green,
**O** vermillion, **OM** reddish-purple. WS3 is the first notebook that actually plots all five.
We strip the top and right spines and drop gridlines by default.

In [ ]:
# Okabe–Ito qualitative palette (colorblind-safe).
OKABE_ITO = {
    "orange":        "#E69F00",
    "sky_blue":      "#56B4E9",
    "bluish_green":  "#009E73",
    "yellow":        "#F0E442",
    "blue":          "#0072B2",
    "vermillion":    "#D55E00",
    "reddish_purple":"#CC79A7",
    "black":         "#000000",
}

# Fixed view -> colour, IDENTICAL to WS1/WS2, now with all five views actually used. C is the
# context-only base; O is the fully ordered headline view; OM adds matchup memory.
VIEW_COLORS = {
    "C":  OKABE_ITO["blue"],
    "U":  OKABE_ITO["orange"],
    "L1": OKABE_ITO["bluish_green"],
    "O":  OKABE_ITO["vermillion"],
    "OM": OKABE_ITO["reddish_purple"],
}
REF_COLOR = OKABE_ITO["black"]  # neutral colour for the count-based reference baselines

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "figure.autolayout": True,
})


def style_axes(ax):
    '''Left+bottom spines only; no top/right. Returns the axis for chaining.'''
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return ax


def new_fig(figsize=(7.2, 4.2)):
    '''One figure, one axis, pre-styled.'''
    fig, ax = plt.subplots(figsize=figsize)
    style_axes(ax)
    return fig, ax

## 2. Data and feature-space EDA — the measurement instrument's raw material

WS3's instrument is the **five nested views**. Before fitting anything we look at what each view
*is*: how wide it gets, which feature groups it adds, and how its within-PA features are missing
early in a plate appearance. The views are information-nested by construction —
`cols(C) ⊆ cols(U) ⊆ cols(O) ⊆ cols(OM)` and `cols(C) ⊆ cols(L1) ⊆ cols(O)` (SPEC §6) — so the
*differences* between them are exactly the sequencing information the ablation measures.

In [ ]:
def load_decision_table(mode, config, seed, n_games):
    '''Return (decision_table, truth_meta) for the chosen world.'''
    if mode == "real":
        if not REAL_TABLE_PATH.exists():
            raise FileNotFoundError(
                f"real decision table not found at {REAL_TABLE_PATH}. "
                "Build it first with `python -m pitchseq.build_table` (RUNBOOK Step 1)."
            )
        return pd.read_parquet(REAL_TABLE_PATH, engine="pyarrow"), {"world": "real"}

    from pitchseq.decision_table import build_decision_table
    from pitchseq.synth import make_null_world, make_positive_world
    if mode == "synth_null":
        raw, truth = make_null_world(n_games=n_games, seed=seed, innings_per_game=6)
    elif mode == "synth_positive":
        raw, truth = make_positive_world(n_games=n_games, seed=seed, innings_per_game=6,
                                         effect_size=0.30, velo_gap_threshold=5.0)
    else:
        raise ValueError(f"unknown DATA_MODE {mode!r}")
    return build_decision_table(raw), truth


table, truth = load_decision_table(DATA_MODE, CONFIG, SEED, N_GAMES)

splits = make_splits(table, CONFIG)["primary"]
train = table.loc[splits["train"].to_numpy()].reset_index(drop=True)
val = table.loc[splits["val"].to_numpy()].reset_index(drop=True)
test = table.loc[splits["test"].to_numpy()].reset_index(drop=True)

print(f"world      : {truth.get('world', DATA_MODE)}")
print(f"total rows : {len(table):,}   columns: {len(table.columns)}")
print(f"train rows : {len(train):,}   (seasons {CONFIG['split']['train']})")
print(f"val rows   : {len(val):,}   (season {CONFIG['split']['val']})")
print(f"test rows  : {len(test):,}   (locked season {CONFIG['split']['test']})")
if DATA_MODE == "synth_positive":
    print(f"planted    : {truth.get('mechanism')} effect={truth.get('effect')} "
          f"threshold={truth.get('threshold')} empirical_whiff_lift={truth.get('empirical_whiff_lift')}")

**How to read the split.** On the synthetic worlds the season is assigned cyclically, so train /
val / test all exist and the temporal-split machinery is exercised end-to-end (SPEC §7); on real
data these are 2021–2023 / 2024 / 2025. Every model below fits on `train` and is scored on `val`,
with the **locked** `test` reserved for the central table's confirmation row only.

In [ ]:
# (a) View widths — the nested feature ladder (committed demo: C=29 U=46 L1=41 O=82 OM=103).
widths, feat_names = {}, {}
for v in VIEWS:
    X, meta = build_view(train, v)
    widths[v] = meta["n_features"]
    feat_names[v] = list(meta["feature_names"])
    del X

fig, ax = new_fig(figsize=(7.2, 4.0))
bars = ax.bar(VIEWS, [widths[v] for v in VIEWS], color=[VIEW_COLORS[v] for v in VIEWS])
for v, b in zip(VIEWS, bars):
    ax.annotate(f"{widths[v]}", (b.get_x() + b.get_width()/2, b.get_height()),
                ha="center", va="bottom", fontsize=10)
ax.set_ylabel("number of features")
ax.set_title("The five nested views widen from context to matchup memory\n"
             "(each view contains the ones to its left; the gaps are the sequencing information)")
plt.show()
print("view widths:", {v: widths[v] for v in VIEWS})

*Caption.* Each view **contains** the ones before it, so the width increments are exactly the
information added: `U` and `L1` layer the unordered bag and the previous pitch onto context `C`;
`O` adds the ordered slots and physics deltas; `OM` adds the matchup block. The ablation reads
the *loss* differences across this ladder; the widths are the *information* differences that drive
them.

In [ ]:
# (b) Feature-group taxonomy — categorise the OM view's columns into the study's groups.
CONTEXT_ROLLING = [c for c in CONTEXT_COLS if c.startswith(("repertoire_mix_", "batter_tend_"))]

def feature_group(col):
    if col in CONTEXT_COLS:
        return "rolling (repertoire/tendency)" if col in CONTEXT_ROLLING else "context"
    if col.startswith("u_"):
        return "unordered (U)"
    if col.startswith("prev_") or col.startswith("l1_"):
        return "L1-token"
    if col.startswith("o_"):
        return "ordered-slots (O)"
    if col.startswith("om_") or "batter_family" in col or col.startswith("bvf_"):
        return "matchup buckets (OM)"
    return "matchup buckets (OM)"  # remaining OM-block rolling cols

GROUPS = ["context", "rolling (repertoire/tendency)", "L1-token", "unordered (U)",
          "ordered-slots (O)", "matchup buckets (OM)"]
GROUP_COLORS = {
    "context": OKABE_ITO["blue"], "rolling (repertoire/tendency)": OKABE_ITO["sky_blue"],
    "L1-token": OKABE_ITO["bluish_green"], "unordered (U)": OKABE_ITO["orange"],
    "ordered-slots (O)": OKABE_ITO["vermillion"], "matchup buckets (OM)": OKABE_ITO["reddish_purple"],
}
counts = {g: [sum(feature_group(c) == g for c in feat_names[v]) for v in VIEWS] for g in GROUPS}

fig, ax = new_fig(figsize=(7.6, 4.4))
bottom = np.zeros(len(VIEWS))
for g in GROUPS:
    ax.bar(VIEWS, counts[g], bottom=bottom, label=g, color=GROUP_COLORS[g])
    bottom += np.array(counts[g])
ax.set_ylabel("features in group")
ax.set_title("Feature-group taxonomy across the five views\n"
             "(context + rolling are shared; each richer view stacks a new group on top)")
ax.legend(frameon=False, fontsize=8, ncol=2)
plt.show()
for v in VIEWS:
    print(f"{v:<3}", {g: counts[g][VIEWS.index(v)] for g in GROUPS if counts[g][VIEWS.index(v)]})

*Caption.* The taxonomy makes the nesting concrete: **context** (count, base-out, inning, score,
handedness) and **rolling** repertoire/tendency features are shared by every view; `U` adds the
order-invariant bag; `L1` the previous-pitch token and its physics diffs; `O` the ordered slots,
consecutive-difference features (velocity/location deltas), and run length; `OM` the within-game
batter-vs-pitcher and trailing batter-vs-family buckets. The **ordered-slots** group is where the
mechanism WS1 was blind to lives (see §8).

In [ ]:
# (c) Missingness-indicator pattern by pitch_number — within-PA history is absent early.
Xom, _ = build_view(train, "OM")
miss_cols = [c for c in Xom.columns if c.endswith("_missing")]
pn = train["pitch_number"].to_numpy()
buckets = [1, 2, 3, 4, 5]
rows = []
for c in miss_cols:
    mvals = Xom[c].to_numpy().astype(float)
    rows.append([float(mvals[pn == b].mean()) if (pn == b).any() else np.nan for b in buckets])
M = np.array(rows)

fig, ax = new_fig(figsize=(7.4, 4.2))
for i, c in enumerate(miss_cols):
    ax.plot(buckets, M[i], "-o", lw=1.8, ms=5, label=c)
ax.set_xlabel("pitch_number in the PA")
ax.set_ylabel("fraction with missing-indicator = 1")
ax.set_xticks(buckets)
ax.set_ylim(-0.03, 1.03)
ax.set_title("Within-PA history is missing early, by construction\n"
             "(pitch 1 has no prior pitch; the indicators switch off as the PA deepens)")
ax.legend(frameon=False, fontsize=7, ncol=2)
plt.show()
del Xom

*Caption + the D11 note.* At pitch 1 there is no prior pitch, so every within-PA history indicator
is on (`u_prior_missing`, `l1_prev_missing`, `o_s1/2/3_missing`, …); as the PA deepens the slots
fill and the indicators switch off (slot 3 needs three priors, so `o_s3_missing` stays on longest).
This is the tabular encoding of ordered history (**decision D11**): the `O` view is `C` + the
last-3 ordered pitch-token **slots** + consecutive-difference and run-length features, neutral-
filled with explicit `*_missing` flags. The tree reads *order* through *which slot* a family sits
in, and reads the *mechanism* through the physics deltas — the same information set a sequence
model uses (WS6), differing only in representation (`THEORY.md` §3). The `first_pitch` slice, where
all these indicators are on, is precisely where only matchup memory (`OM`) can add anything.

## 3. The two stacks (exposition)

WS3 fits two things per view; the exact formulas are in `THEORY.md` (§§2, 4) and match the code in
`model.py` line-for-line. This section is exposition — no fitting yet.

### The behavior model `μ(a | s)`

A multiclass LightGBM over the 8 pitch families, trained on the state view **alone** — no action,
no execution. It answers *selection* (finding #1): does ordered history sharpen the next-pitch
guess? It is leakage-audited (SPEC §0): no current-pitch execution column may be a feature.

### The decomposed event-tree outcome stack (decision D32)

Three separately-fitted pieces, each **conditioned on the current action** (decision D22 —
`action_family` appended to the view, SPEC §0's "after conditioning on the current pitch," which
keeps finding #1 out of finding #2):

- **Stage A** `P(o₁ | s, a)` — a LightGBM over the 6 level-1 outcomes
  `(ball, called_strike, whiff, foul, hbp, in_play)`.
- **Stage B** `P(o₂ | s, a)` — a LightGBM over the 5 level-2 outcomes
  `(single, double, triple, home_run, out_or_other)`, trained on the **in-play rows only**.
- **Stage C** `V(node, c)` — count-conditional node run values: the train-fold mean reward of each
  event-tree node at each count, with a global-node fallback.

### The assembly (the exact D32 formula)

With the count `c = (balls, strikes)`, the 10 leaves are the 5 terminal level-1 outcomes plus the
5 level-2 refinements of `in_play`, and the expected reward is the leaf-probability-weighted sum
of node values:

$$\mathbb{E}[R \mid s, a, c] =
  \sum_{k \in \{\text{ball},\text{called\_strike},\text{whiff},\text{foul},\text{hbp}\}}
       P_A(o_1 = k \mid s, a)\; V_1(k, c)
  \;+\; P_A(o_1 = \text{in\_play} \mid s, a)
       \sum_{j \in \mathcal{O}_2} P_B(o_2 = j \mid s, a)\; V_2(j, c).$$

The predictive uncertainty is residual-based (no bootstrap), by the law of total variance over the
leaves:

$$\mathrm{Var}[R \mid s, a, c] = \sum_{\ell} p_\ell\big(S_\ell(c) + V_\ell(c)^2\big) - \mathbb{E}[R \mid s, a, c]^2,
\qquad \texttt{exp\_reward\_sd} = \sqrt{\max(\mathrm{Var}, 0)},$$

with `p_ℓ` the leaf probability, `V_ℓ` its node value, and `S_ℓ` its train-fold within-node reward
variance. These are the two boxed formulas of `THEORY.md` §4.

### The cross-check and the budget

A **direct** LightGBM regressor `Ê_dir[R | s, a]` on the same `(view + action)` features gives a
consistency check: `mean |E_decomposed − E_direct|`, flagged above `0.03` on the `|R| ~ 0.05–0.3`
scale (decision D32). Hyperparameters follow the **equal-budget** discipline (decision D34): the
predeclared defaults plus an optional identical 12-combo grid per view, selected on a holdout
carved from the **train** seasons — so a loss gap between views isolates *information*, not search
budget (`THEORY.md` §9). This notebook uses light demo params (`NB_PARAMS`); the CLI uses the full
budget with `--tune`.

## 4. The behavior stack — `μ(a | s)` per view

We fit the behavior model on `train` for each view, score next-pitch **selection** log loss on
`val` through the shared harness, and compare against the four count-based reference baselines
(`eval/baselines.py`). The `O` view should beat the `pitcher_count_prev` reference; a
`{PLACEHOLDER}` row is left for the WS1-table and WS2-grammar selection losses (their own RUNBOOK
steps) so the three workstreams can be read side by side.

In [ ]:
def behavior_pred_df(model, rows, view):
    '''Standard-schema action_probs prediction frame for the shared harness.'''
    proba = model.predict_proba(rows)
    df = pd.DataFrame({"row_id": rows["row_id"].to_numpy()})
    for j, col in enumerate(ACTION_PROB_COLS):
        df[col] = proba[:, j]
    df["model_id"] = "ws3_gbdt_stack"
    df["state_view"] = view
    df["seconds"] = 0.0
    df["peak_mem_mb"] = 0.0
    df["n_params"] = int(model.n_params)
    return df

behavior, beh_val_pred, beh_sel_loss = {}, {}, {}
for v in VIEWS:
    behavior[v] = BehaviorModel(v).fit(train, tune=False, params=NB_PARAMS, seed=SEED)
    beh_val_pred[v] = behavior_pred_df(behavior[v], val, v)
    rep = evaluate_predictions(beh_val_pred[v], val, config=CONFIG, n_boot=N_BOOT, seed=SEED)
    beh_sel_loss[v] = rep["slices"]["all"]["action_prob"]["log_loss"]

# Count-based reference baselines, scored through the same harness (selection log loss).
from pitchseq.eval.baselines import build_baselines
ref_loss = {}
for name, mdl in build_baselines(alpha=8.0).items():
    mdl.fit(train)
    proba = mdl.predict_proba(val)
    df = pd.DataFrame({"row_id": val["row_id"].to_numpy()})
    for j, col in enumerate(ACTION_PROB_COLS):
        df[col] = proba[:, j]
    df["model_id"] = f"baseline_{name}"; df["state_view"] = "NA"
    df["seconds"] = 0.0; df["peak_mem_mb"] = 0.0; df["n_params"] = 0
    r = evaluate_predictions(df, val, config=CONFIG, n_boot=N_BOOT, seed=SEED)
    ref_loss[name] = r["slices"]["all"]["action_prob"]["log_loss"]

print("selection log loss (val), WS3 behavior model per view:")
for v in VIEWS:
    ref = "pitcher_count" if v == "C" else "pitcher_count_prev"
    flag = "<= ref" if beh_sel_loss[v] <= ref_loss[ref] else "> ref"
    print(f"  {v:<3}: {beh_sel_loss[v]:.4f}   ref[{ref}]={ref_loss[ref]:.4f}  ({flag})")
print("  references:", {k: round(x, 4) for k, x in ref_loss.items()})
print("  WS1-table / WS2-grammar selection loss (val): {PLACEHOLDER} / {PLACEHOLDER}")

**How to read it.** WS3's boosted behavior model should sit **at or below** the
`pitcher_count_prev` reference on the history views (`U/L1/O/OM`) and be competitive with WS2's
grammar on selection — the GBDT is not expected to *lose* the selection channel to a shrunk count
table. On the null synthetic world the `O` behavior model beats every count reference (completed
validation). The `{PLACEHOLDER}` WS1/WS2 cells are filled from their own runs so the ladder's three
selection instruments read side by side; a boosted model that only ties the grammar on selection
still earns its place by the *outcome* stack the tabular workstreams could not provide.

In [ ]:
# Calibration of the O behavior model (top-class reliability curve).
proba_O = behavior["O"].predict_proba(val)
conf = proba_O.max(axis=1)
pred_fam = np.array(FAMILIES)[proba_O.argmax(axis=1)]
correct = (pred_fam == val["family"].astype("object").to_numpy()).astype(float)
rel = reliability_table(correct, conf, n_bins=10)

fig, ax = new_fig(figsize=(5.4, 5.2))
ax.plot([0, 1], [0, 1], ls="--", color=OKABE_ITO["black"], lw=1, label="perfect calibration")
ax.plot(rel["mean_pred"], rel["frac_pos"], "-o", color=VIEW_COLORS["O"], lw=2)
for _, r in rel.iterrows():
    ax.annotate(f"n={int(r['count'])}", (r["mean_pred"], r["frac_pos"]),
                textcoords="offset points", xytext=(4, -9), fontsize=7, color="gray")
ax.set_xlabel("mean predicted confidence (top class)")
ax.set_ylabel("observed hit rate")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("Is the O behavior model calibrated?\n(points on the dashed line = confidence matches reality)")
ax.legend(frameon=False, fontsize=9)
plt.show()

*Caption.* Boosted trees rank well but are not automatically calibrated (Niculescu-Mizil and
Caruana, 2005): points sagging below the diagonal are over-confidence in that bin. Bin counts are
annotated so sparse bins are discounted. Selection calibration matters less downstream than the
*outcome* calibration (§5), which the prescriptive workstreams consume as run-value *levels*.

In [ ]:
# Top-15 gain features per view — small multiples (behavior model gain importances).
fig, axes = plt.subplots(2, 3, figsize=(13.5, 8.2))
axes = axes.ravel()
for i, v in enumerate(VIEWS):
    gains = behavior[v].feature_gain()  # {feature: gain}, descending
    top = list(gains.items())[:15]
    names = [k for k, _ in top][::-1]
    vals = [g for _, g in top][::-1]
    ax = style_axes(axes[i])
    ax.barh(range(len(names)), vals, color=VIEW_COLORS[v])
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=6.5)
    ax.set_title(f"{v}  (top-15 gain)", color=VIEW_COLORS[v])
    ax.tick_params(axis="x", labelsize=7)
axes[-1].axis("off")
fig.suptitle("What each behavior model splits on (gain importance)\n"
             "count + pitcher/repertoire should dominate C; history views add slot/transition features",
             fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

**How to read the small multiples (a sanity check).** The `C` view has no pitch history, so its
top splits **should** be count (`balls`, `strikes`), pitcher fatigue/exposure
(`pitcher_pitch_count`, `times_thru_order`), and the rolling `repertoire_mix_*` / `batter_tend_*`
features — if something else dominated `C`, the feature build would be suspect. The history views
(`U/L1/O/OM`) should show those context features *plus* the added group: `prev_family` for `L1`,
the `u_count_*` for `U`, the `o_s*_family` slots for `O`, the `om_*` buckets for `OM`. This is the
qualitative check that each view is reading the information it is supposed to.

## 5. The outcome stack — decomposed `E[R | s, a]` per view

Now the finding-#2 machinery. For each view we fit the decomposed stack (stage A, stage B, the
count-conditional node values, and the direct-regressor cross-check), score the node-level losses
and run-value MAE/RMSE + calibration on `val`, and report the decomposed-vs-direct agreement.

In [ ]:
def outcome_pred_df(model, rows, view):
    '''Standard-schema outcome1/outcome2/exp_reward prediction frame.'''
    p1 = model.predict_outcome1(rows)
    p2 = model.predict_outcome2(rows)
    er, sd = model.exp_reward(rows, with_sd=True)
    df = pd.DataFrame({"row_id": rows["row_id"].to_numpy()})
    for j, col in enumerate(OUTCOME1_PROB_COLS):
        df[col] = p1[:, j]
    for j, col in enumerate(OUTCOME2_PROB_COLS):
        df[col] = p2[:, j]
    df["exp_reward"] = er
    df["exp_reward_sd"] = sd
    df["model_id"] = "ws3_gbdt_stack"; df["state_view"] = view
    df["seconds"] = 0.0; df["peak_mem_mb"] = 0.0; df["n_params"] = int(model.n_params)
    return df

outcome, out_val_pred, out_blocks, disagree = {}, {}, {}, {}
for v in VIEWS:
    outcome[v] = OutcomeStack(v).fit(train, tune=False, params=NB_PARAMS, seed=SEED)
    out_val_pred[v] = outcome_pred_df(outcome[v], val, v)
    rep = evaluate_predictions(out_val_pred[v], val, config=CONFIG, n_boot=N_BOOT, seed=SEED)
    out_blocks[v] = rep["slices"]["all"]
    disagree[v] = outcome[v].disagreement(val)

print("outcome stack (val): node log loss + run value")
print(f"  {'view':<4} {'o1_ll':>8} {'o2_ll':>8} {'rv_mae':>8} {'rv_rmse':>8} {'cal_slope':>10} {'cal_int':>9}")
for v in VIEWS:
    o1 = out_blocks[v]["outcome1_prob"]["log_loss"]
    o2b = out_blocks[v].get("outcome2_prob", {})
    o2 = o2b.get("log_loss", float("nan"))
    rvb = out_blocks[v].get("exp_reward", {})
    mae = rvb.get("mae", float("nan")); rmse = rvb.get("rmse", float("nan"))
    cal = rvb.get("calibration", {})
    print(f"  {v:<4} {o1:8.4f} {o2:8.4f} {mae:8.4f} {rmse:8.4f} "
          f"{cal.get('slope', float('nan')):10.3f} {cal.get('intercept', float('nan')):+9.4f}")

**How to read it.** The outcome-1 log loss is the node-level analogue of the selection loss; the
`O`/`OM` rows are where any ordered/matchup *outcome* effect shows up (quantified with a CI in §6).
Run value is a change in run expectancy — a small number in runs — so the MAE is small by nature;
what matters is the **calibration slope** (want ≈ 1) and **intercept** (want ≈ 0): the prescriptive
workstreams consume the run-value *levels*, so a slope far from 1 would distort every downstream
policy value. The outcome-2 loss is over the in-play refinements only.

In [ ]:
# Decomposed-vs-direct agreement (the D32 cross-check), per view.
print(f"decomposed vs direct  mean|E_dec - E_dir|  (flag threshold = {DISAGREEMENT_FLAG_ABS})")
for v in VIEWS:
    d = disagree[v]
    tag = "  <-- FLAG" if d["flag"] else ""
    print(f"  {v:<4} mean={d['mean_abs_diff']:.4f}  max={d['max_abs_diff']:.4f}{tag}")
flagged = [v for v in VIEWS if disagree[v]["flag"]]
print("flagged:", flagged if flagged else "none (all within tolerance)")

**The honest OM flag story.** The cross-check compares the interpretable event-tree assembly
against a black-box regressor on the same features. On the **untuned demo models** the committed
validation showed `OM` landing *slightly* over the 0.03 threshold while the other four views sat at
0.023–0.026 (unflagged) — carried forward as an honest flag (dispatch log). The reading (`THEORY.md`
§5): `OM`'s larger, sparser feature space fragments the stage-A/B fits relative to the count-
conditional node values (which carry no matchup dimension), nudging the two estimates apart — a mild
mis-alignment expected to tighten under `--tune`, not a correctness failure. It is the same
fragmentation story the negative `Δ_matchup` tells from the loss side (§6). (At this notebook's demo
scale the exact flag set can differ run to run; the committed-scale behaviour is the one to cite.)

## 6. The central table — `Δ_order` and `Δ_matchup`, three targets, with clustered CIs

This is the heart of the project. For **selection**, **outcome-1**, and **run-value MAE** we form
the SPEC §6 ablations with pitcher-game clustered confidence intervals, on validation and — for the
outcome target — the **locked test**. `Δ_order` is read under the D21 rule; `Δ_matchup` is read
directly (no `min`-bias). Every prior workstream was building toward this table.

In [ ]:
sel_pred = {v: beh_val_pred[v] for v in VIEWS}
out_pred = {v: out_val_pred[v] for v in VIEWS}

cv_sel = compare_views(sel_pred, val, config=CONFIG, target="family", n_boot=N_BOOT, seed=SEED)
cv_o1 = compare_views(out_pred, val, config=CONFIG, target="outcome1", n_boot=N_BOOT, seed=SEED)
rv = _runvalue_compare(out_pred, val, CONFIG, N_BOOT, SEED)   # run-value MAE ablation (CLI-exact)

def line(name, do, do_ci, dm, dm_ci):
    def ci(c): return f"[{c['lo']:+.4f}, {c['hi']:+.4f}]" if c else "[   n/a   ]"
    print(f"  {name:<12} Delta_order={do:+.4f} {ci(do_ci):<22}   "
          f"Delta_matchup={('%+.4f' % dm) if dm is not None else '  n/a  '} {ci(dm_ci)}")

print("CENTRAL ABLATION (validation; clustered 95% CI):")
line("selection", cv_sel["deltas"]["delta_order"], cv_sel["delta_order_ci"],
     cv_sel["deltas"].get("delta_matchup"), cv_sel.get("delta_matchup_ci"))
line("outcome-1", cv_o1["deltas"]["delta_order"], cv_o1["delta_order_ci"],
     cv_o1["deltas"].get("delta_matchup"), cv_o1.get("delta_matchup_ci"))
line("run-value", rv["delta_order"], rv["delta_order_ci"],
     rv.get("delta_matchup"), rv.get("delta_matchup_ci"))
print("  (Delta_order = min[U,L1]-O ; Delta_matchup = O-OM ; positive = the richer view helps)")

**The reading rules (binding).**

- **`Δ_order`, read under D21.** `Δ_order = Loss(min[U,L1]) − Loss(O)` takes the *minimum* of two
  noisy losses, so it is **negatively biased under the null** (`THEORY.md` §6.2). The criterion is
  **significantly positive** (CI lower bound > 0); a small **negative** value reads as *consistent
  with no ordering effect*, **never** "order hurts." The **outcome-1** `Δ_order` is the project's
  central number (finding #2); the **selection** `Δ_order` is finding #1, a separate claim.
- **`Δ_matchup`, read directly — the NO-min-bias note.** `Δ_matchup = Loss(O) − Loss(OM)` takes
  **no minimum**, so it carries **no** optimism bias: read its CI at face value. A significantly
  **negative** `Δ_matchup` therefore means the matchup features genuinely **cost** out-of-sample —
  a real fragmentation/overfit cost of ~21 sparse columns that add variance without signal
  (`THEORY.md` §6.3), *not* a statistical artifact. This is exactly what is observed on both
  synthetic worlds (§8) and is pre-written as the `M−` branch (§9).

In [ ]:
# Locked-test confirmation row (outcome-1 Delta_order on the unseen 2025-equivalent test season).
out_pred_test = {v: outcome_pred_df(outcome[v], test, v) for v in VIEWS}
cv_o1_test = compare_views(out_pred_test, test, config=CONFIG, target="outcome1",
                           n_boot=N_BOOT, seed=SEED)
do_t = cv_o1_test["deltas"]["delta_order"]; ci_t = cv_o1_test["delta_order_ci"]
dm_t = cv_o1_test["deltas"].get("delta_matchup"); dmci_t = cv_o1_test.get("delta_matchup_ci")
print("LOCKED TEST (outcome-1):")
print(f"  Delta_order   = {do_t:+.4f}  CI[{ci_t['lo']:+.4f}, {ci_t['hi']:+.4f}]")
if dm_t is not None:
    print(f"  Delta_matchup = {dm_t:+.4f}  CI[{dmci_t['lo']:+.4f}, {dmci_t['hi']:+.4f}]")

In [ ]:
# Per-slice outcome-1 Delta_order (SPEC §6 slices). first_pitch isolates matchup memory.
slice_names = list(CONFIG.get("report_slices", ["all"]))
masks = slice_masks(val, slice_names)
print("per-slice outcome-1 Delta_order (min[U,L1]-O):")
for sname in slice_names:
    mask = masks.get(sname)
    if mask is None or int(mask.sum()) < 200:
        print(f"  {sname:<12}: n/a (n={0 if mask is None else int(mask.sum())})")
        continue
    sub_val = val.loc[mask].reset_index(drop=True)
    keep = set(sub_val["row_id"].tolist())
    sub_pred = {v: out_pred[v][out_pred[v]["row_id"].isin(keep)] for v in VIEWS}
    try:
        cvs = compare_views(sub_pred, sub_val, config=CONFIG, target="outcome1",
                            n_boot=max(20, N_BOOT // 2), seed=SEED)
        do_s = cvs["deltas"]["delta_order"]; ci_s = cvs["delta_order_ci"]
        print(f"  {sname:<12}: {do_s:+.4f}  CI[{ci_s['lo']:+.4f}, {ci_s['hi']:+.4f}]  (n={int(mask.sum())})")
    except (KeyError, ValueError) as e:
        print(f"  {sname:<12}: n/a ({type(e).__name__})")

**How to read the slices.** A *genuine* ordered outcome effect should **concentrate** in
`long_pa` (t ≥ 3) and `two_strike` — the counts and depths where ordered history exists and matters
— rather than appear only in aggregate; that per-slice consistency is one of the three checks (with
the mechanism ablation and the locked-test row) before any H1 headline. The **`first_pitch`** slice
is special: on the first pitch there is *no* within-PA history, so `O` cannot differ from `L1`/`U`
there — the slice **isolates matchup memory by construction**, and it is the cleanest witness for
the matchup axis (a genuine `M+` must show up here; a `first_pitch` `Δ_matchup ≤ 0` while the
aggregate is positive would be a red flag).

## 7. The q̂ grid and propensities — what the prescriptive phase consumes (D33)

WS3 publishes two derived artifacts per view (decision D33): the **counterfactual value grid**
`q̂(s, a) = E[R | s, A=f]` for all 8 families, and the behavior **propensities** `μ(a | s)`. We show
one concrete decision row — its q̂ across all 8 families, the feasibility mask, and the observed
propensities — because *this row* is exactly what WS4's bandit, WS5's MDP, and WS7's OPE read.

In [ ]:
# One concrete decision row (a sequence-eligible one so the ordered view is engaged).
elig = np.flatnonzero((val["pitch_number"].to_numpy() >= 3))
row_i = int(elig[len(elig) // 2]) if len(elig) else 0
row = val.iloc[[row_i]]

qhat = outcome["O"].q_grid(row)[0]          # E[R | s, A=f] for all 8 families
mu = behavior["O"].predict_proba(row)[0]    # behavior propensities mu(a|s)

# Feasibility mask: the production leakage-safe mask, with a train-repertoire fallback on synth.
try:
    fmask = feasible_action_mask(val)
    feas = fmask.iloc[row_i][[f"feasible_{f}" for f in FAMILIES]].to_numpy().astype(bool)
except Exception:
    pit = val.iloc[row_i]["pitcher"]
    thrown = set(train.loc[train["pitcher"] == pit, "family"].astype("object"))
    feas = np.array([f in thrown for f in FAMILIES])

print(f"decision row_id = {val.iloc[row_i]['row_id']}  "
      f"(count {int(row['balls'].iloc[0])}-{int(row['strikes'].iloc[0])}, "
      f"pitch_number {int(row['pitch_number'].iloc[0])})")
print(f"  {'family':<8} {'qhat E[R|s,a]':>14} {'mu(a|s)':>9} {'feasible':>9}")
for j, f in enumerate(FAMILIES):
    print(f"  {f:<8} {qhat[j]:>14.4f} {mu[j]:>9.3f} {str(bool(feas[j])):>9}")

fig, ax = new_fig(figsize=(7.6, 4.2))
x = np.arange(len(FAMILIES))
colors = [VIEW_COLORS["O"] if feas[j] else "#cccccc" for j in range(len(FAMILIES))]
ax.bar(x, qhat, color=colors)
ax2 = ax.twinx()
ax2.plot(x, mu, "o-", color=OKABE_ITO["black"], lw=1.5, label="mu(a|s)")
ax2.set_ylabel("behavior propensity mu(a|s)")
ax.set_xticks(x); ax.set_xticklabels(FAMILIES)
ax.set_ylabel("q-hat  E[R | s, A=f]  (runs)")
ax.set_title("One decision's what-if card: q̂ for all 8 families (grey = infeasible)\n"
             "black line = what the pitcher actually tends to throw (propensity)")
ax2.legend(frameon=False, fontsize=9, loc="upper right")
plt.show()

**This row is what WS4/WS5/WS7 consume (decision D33).** The q̂ bar is the expected run value of
each pitch *if it were thrown here*; the grey bars are families the feasibility mask rules out
(this pitcher does not throw them enough to recommend, SPEC §4); the black line is the behavior
model's estimate of what he *actually* tends to throw. WS4's bandit picks a target policy from q̂
restricted to feasible actions; WS5's MDP uses the node values behind q̂ to price a *setup* pitch;
WS7's OPE weights by `μ(a|s)` and refuses to answer when its estimators disagree. WS3 saves these
per view with `load_ws3_artifacts` so none of them refit an outcome model. **The firewall:** q̂ is
`E[R | s, A=f]`, a *prediction* under a hypothetical pitch — not `E[R | s, do(A=f)]`, a causal
effect (`THEORY.md` §8). It becomes a *recommendation* only through the OPE gate.

In [ ]:
# Counterfactual sensitivity sanity: swapping the action must move the outcome probabilities.
p_obs = outcome["O"].predict_outcome1(row)[0]
alt = "SL" if FAMILIES[int(np.argmax(mu))] != "SL" else "FF"
p_alt = outcome["O"].predict_outcome1(row, action=alt)[0]
print(f"outcome-1 probs under observed vs counterfactual action ({alt}):")
print(f"  {'outcome1':<14} {'observed':>9} {'A='+alt:>9} {'delta':>9}")
for j, o in enumerate(OUTCOME1):
    print(f"  {o:<14} {p_obs[j]:>9.3f} {p_alt[j]:>9.3f} {p_alt[j]-p_obs[j]:>+9.3f}")
moved = float(np.abs(p_alt - p_obs).sum())
print(f"total |change| in outcome-1 distribution: {moved:.3f}  "
      f"({'responds to the action (expected)' if moved > 1e-3 else 'no response — investigate'})")

*Caption.* Swapping the conditioned action from what was thrown to a counterfactual family moves
the predicted pitch-result distribution — the outcome model genuinely **conditions on the action**
(decision D22), which is what makes the q̂ grid non-trivial. This is a *sensitivity* check that the
machinery responds to the action feature; it is **not** a causal claim (the same firewall).

## 8. Falsification — the D35 real-model test on both synthetic worlds

WS3 reruns the SPEC §11 oracle as a real-model test (decision D35). The **committed validation
numbers below are completed results**, stated as done; the live cell reproduces the signatures at
the demo budget (`FALS_*`). The falsification battery is **synthetic-only by design** — its
permutation refits are infeasible on 3.85M rows — so on real data the central table (§6) carries
the ablation and these worlds carry the recovery/permutation checks.

In [ ]:
# Live D35 battery on both worlds at the committed-validation scale (FALS_* in the setup cell).
# This refits all five views + permutations per world, so it is ~11 min PER world — the study's
# heaviest cell. The COMMITTED numbers stated in the markdown below are the ones to cite; lower
# FALS_N_GAMES for a quick (verdict-approximate) pass.
fals = {}
for world in ("null", "positive"):
    res = run_ws3(synth=world, n_games=FALS_N_GAMES, seed=7, n_boot=30,
                  n_perm=FALS_N_PERM, ci_boot=FALS_CI_BOOT, write_outputs=False,
                  fals_params={"n_estimators": 80})
    fals[world] = res["falsification"]

for world in ("null", "positive"):
    f = fals[world]
    acc = f["acceptance"]
    print("=" * 66)
    print(f" {world.upper()} world  ->  D35 verdict: {f['d35_verdict']}")
    print(f"   outcome-1 delta_order = {acc['delta_order']:+.4f} "
          f"CI[{acc['delta_order_ci']['lo']:+.4f}, {acc['delta_order_ci']['hi']:+.4f}]  "
          f"perm p={acc['permutation_p']:.3f}  fired={acc['permutation_fired']}")
    print(f"   mechanism top group   = {f['mechanism_ablation']['top_group']}")
    if world == "positive":
        print(f"   recovered whiff-lift  = {f['recovered_whiff_lift']:.4f}  "
              f"recovery ratio = {f['recovery_ratio']:.3f}  (floor {f['recovery_ratio_floor']})")
print("=" * 66)

**Completed synthetic validation (state these as done — the committed WS3a run).**

- **Null world → `NULL_QUIET`.** The D20 finding-#1/finding-#2 separation, displayed live: the
  *selection* `Δ_order` is **+0.0088** (CI [+0.0021, +0.0143], significantly positive — the planted
  order-2 no-three-in-a-row habit *is* detected, finding #1), while the *outcome* `Δ_order` is
  **−0.0033** (CI [−0.0081, +0.0008], not significant — no outcome order effect, finding #2 correctly
  absent). The permutation test does not fire (**p = 0.880**), and the mechanism ablation's top group
  is `family_slots` (there is no velocity mechanism to find). `Δ_matchup` (outcome) is **−0.0078**
  (CI [−0.0145, −0.0018]) — significantly negative even here, the fragmentation cost (§6, §9 M−).
- **Positive world → `MECHANISM_RECOVERED`.** The outcome `Δ_order` is **+0.0277** (CI [+0.0211,
  +0.0339]) on validation and **+0.0382** (CI [+0.0313, +0.0449]) on the locked test; the permutation
  test fires (**p = 0.040**); the mechanism ablation isolates the velocity channel (`velo_diff` top,
  Δ ≈ 0.0144); and the model recovers the planted whiff lift at **0.294 of a planted 0.308 — recovery
  ratio 0.955**. `Δ_matchup` (outcome) is **−0.0129** (CI [−0.0193, −0.0080]).

(At this notebook's demo budget the live numbers above approximate these; the committed-scale run is
the one to cite in the paper.)

In [ ]:
# THE CONTRAST EXHIBIT — recovery ratio WS3 vs WS1 (decision D35).
ws3_recovery = fals["positive"].get("recovery_ratio", np.nan)   # live demo value
WS3_COMMITTED, WS1_COMMITTED = 0.955, 0.03                       # committed validation
fig, ax = new_fig(figsize=(6.6, 4.2))
bars = ax.bar(["WS3 (O sees velo delta)", "WS1 (name-keyed table)"],
              [WS3_COMMITTED, WS1_COMMITTED],
              color=[VIEW_COLORS["O"], OKABE_ITO["sky_blue"]])
for b, val_ in zip(bars, [WS3_COMMITTED, WS1_COMMITTED]):
    ax.annotate(f"{val_:.0%}", (b.get_x() + b.get_width()/2, b.get_height()),
                ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_ylabel("fraction of the planted velocity effect recovered")
ax.set_ylim(0, 1.05)
ax.set_title("The contrast exhibit: mechanism visibility is everything\n"
             "same planted effect, same data — the difference is representation")
plt.show()
print(f"committed: WS3 recovery ratio {WS3_COMMITTED:.3f}  vs  WS1 attenuation {WS1_COMMITTED:.3f}")
print(f"live demo: WS3 recovery ratio {ws3_recovery:.3f}")

**Why the contrast (mechanism visibility).** Both models face the *same* planted effect — a whiff
boost when the velocity transition into the previous pitch exceeds 5 mph — on the *same* data.
WS3's `O` view carries `o_velo_delta_last` as a **native feature**, so a single tree split
`o_velo_delta_last ≥ 5` represents the mechanism directly, and WS3 recovers **~95%** of it. WS1's
tables are keyed on pitch **names** and see velocity only through the name → velocity-band proxy, so
they recover **~3%**. Said plainly: *when the model can see the actual mechanism, it recovers 95% of
it; when it can only see pitch names, 3%.* That single contrast is the study's concrete argument for
why the ladder needs more than one rung.

In [ ]:
# Top-gain-feature evidence: o_velo_delta_last is the top gain feature of the O / OM outcome models.
print("outcome stack (stage A) top gain features — is the velo mechanism the top split?")
for v in ("O", "OM"):
    gains = outcome[v].feature_gain()
    top = list(gains.items())[:6]
    rank = [k for k, _ in list(gains.items())]
    pos = rank.index("o_velo_delta_last") + 1 if "o_velo_delta_last" in rank else None
    print(f"  {v}: top-6 = {[k for k, _ in top]}")
    print(f"       o_velo_delta_last rank = {pos}  "
          f"(committed positive-world run: TOP feature, ~28,700 gain)")

*Caption.* On the committed positive-world run, `o_velo_delta_last` — the velocity change into the
previous pitch — is the **single top gain feature** of both the `O` and `OM` stage-A outcome models
(≈28,700 gain), the direct fingerprint of the recovered mechanism. On this notebook's small demo the
exact rank can vary, but the ordered velocity-delta feature should sit near the top of the `O`/`OM`
outcome models whenever the planted effect is present.

## 9. Results — branched interpretation (the SPEC §6 grid)

The project's headline results section. The result is read on **two independent axes**, exactly the
SPEC §6 interpretation grid: an **order** axis (does `O` beat the history-lite views on *outcomes*?)
with branches **H1/H2/H3**, read under D21; and a **matchup** axis (does `OM` beat `O`?) with
branches **M+/M0/M−**, read directly. A code cell inspects the computed outcome-1 ablation and prints
which branch fired on each axis; the pre-written markdown below holds the interpretation for every
branch. Because the axes are independent, any H×M pair is a coherent reading — a short cross-reading
closes the section.

In [ ]:
# --- Order axis: H1 (O beats U and L1) / H2 (history beats C, order doesn't) / H3 (nothing beats C) ---
o1_loss = {v: out_blocks[v]["outcome1_prob"]["log_loss"] for v in VIEWS}
SEL_MATERIAL = 3e-3   # log-loss (nats): a small-but-real outcome sharpening vs C
best_hist_gain = max(o1_loss["C"] - o1_loss["U"], o1_loss["C"] - o1_loss["L1"],
                     o1_loss["C"] - o1_loss["O"])
do_o1 = cv_o1["deltas"]["delta_order"]; do_ci = cv_o1["delta_order_ci"]
do_significant = do_ci["lo"] > 0

if best_hist_gain <= SEL_MATERIAL:
    order_branch = "H3"            # nothing beats C
elif do_significant:              # O beats min[U,L1] by more than the clustered CI
    order_branch = "H1"
else:
    order_branch = "H2"           # history beats C, but O ~ U/L1

# --- Matchup axis: M+ (OM beats O) / M0 (no effect) / M- (OM costs) ---
dm_o1 = cv_o1["deltas"].get("delta_matchup"); dm_ci = cv_o1.get("delta_matchup_ci")
if dm_ci is None or dm_o1 is None:
    matchup_branch = "M0"
elif dm_ci["lo"] > 0:
    matchup_branch = "M+"
elif dm_ci["hi"] < 0:
    matchup_branch = "M-"
else:
    matchup_branch = "M0"

print("=" * 68)
print(" WS3 RESULTS — branch selector (outcome-1 target)")
print("=" * 68)
print(f" world / mode          : {truth.get('world', DATA_MODE)}")
print(f" outcome-1 log loss    : " + "  ".join(f"{v}={o1_loss[v]:.4f}" for v in VIEWS))
print(f" best history gain vs C: {best_hist_gain:+.4f}   (material {SEL_MATERIAL:+.4f})")
print(f" Delta_order (outcome) : {do_o1:+.4f}  CI[{do_ci['lo']:+.4f}, {do_ci['hi']:+.4f}]")
dm_txt = f"{dm_o1:+.4f}  CI[{dm_ci['lo']:+.4f}, {dm_ci['hi']:+.4f}]" if dm_o1 is not None else "n/a"
print(f" Delta_matchup (outcome): {dm_txt}")
print("-" * 68)
print(f" ORDER BRANCH          : {order_branch}")
print(f" MATCHUP BRANCH        : {matchup_branch}")
print("=" * 68)
print(" -> read the matching branch write-ups in the markdown below.")

### Order branch H1 — *`O` beats both `U` and `L1`: genuine ordered outcome dependence*

`Δ_order` (outcome) clears zero (CI lower bound > 0): the fully ordered view lowers outcome log
loss below the better of `U`/`L1` by more than the clustered CI — genuine out-of-sample *sequencing
value* (finding #2), the strong result. Read the effect size against the **+0.0277 synthetic
benchmark**: a real-data `Δ_order` an order of magnitude smaller is meaningful but modest. Before
any headline, clear three checks — per-slice consistency (concentrate in `long_pa` / `two_strike`,
not aggregate only), the mechanism ablation (which feature group carries it), and the **locked-test**
row (does 2025 confirm 2024). If it survives, WS3 hands WS4/WS5 a q̂ grid that *encodes* the ordered
effect for a target policy, and WS7's OPE the task of proving it is also *exploitable* (finding #3) —
the step the firewall forbids WS3 from taking itself.

### Order branch H2 — *history matters, order does not (`O ≈ U/L1`, both beat `C`)*

The history views lower outcome loss below context-only, but `O` does not refine on the better of
`U`/`L1` (`Δ_order` not significantly positive). Within-PA history carries outcome value; its *order*
beyond the previous pitch / unordered bag does not — **SPEC §13's explicitly anticipated result**
("O barely beats L1"), consistent with the motif literature, and reported without embarrassment
(SPEC §0). It tells the prescriptive phase it need carry only `L1`/`U` history and sets the bar
WS6's learned representation must clear to justify going deeper. The synthetic validation is the
guardrail: recovery ratio 0.955 proves the model *would* have seen deeper order had it existed, so
H2 is a real bound, not a blind spot.

### Order branch H3 — *no sequencing signal in outcomes at all (nothing beats `C`)*

No history view lowers outcome loss below context-only: within-PA history carries no *outcome* value
a strong tabular model resolves — the cleanest finding-#2 null. This does **not** deny finding #1:
selection structure may still exist (WS2's grammar and WS3's own selection `Δ_order` can be positive
while the outcome ablation is null — exactly the D20 separation the null world displays live in §8).
Audit before accepting: confirm the outcome models are calibrated (§5) so a real effect is not masked,
and check whether any selection edge simply does not translate to outcomes. Under H3 the prescriptive
phase inherits a context-only outcome model, and the honest headline is "pitch sequences are
forecastable (WS2) but not, at this data scale, outcome-predictive (WS3)."

### Matchup branch M+ — *`OM` beats `O`: real batter–pitcher adaptation*

`Δ_matchup` clears zero (CI lower bound > 0): the matchup view lowers outcome loss below `O` — the
study's first evidence of longer-term batter–pitcher adaptation carrying outcome value (WS1/WS2 could
not fit `OM`). It must corroborate on the **`first_pitch` slice**, which isolates matchup memory by
construction (no within-PA history exists there, so any `OM`-over-`O` edge is *purely* cross-PA). If
it holds, WS3 supplies a real matchup outcome signal for WS4/WS5/WS7 to exploit and WS6's cross-PA
representation to extend.

### Matchup branch M0 — *no detectable matchup memory (`OM ≈ O`)*

`Δ_matchup` spans zero: the matchup view neither helps nor hurts beyond `O`. No detectable
longer-term batter–pitcher adaptation in outcomes at this data scale — a clean read, and the
expected one if within-game/season rematch counts are too thin to resolve. The prescriptive phase
can use `O` and `OM` interchangeably for outcomes.

### Matchup branch M− — *matchup features actively cost out-of-sample (`OM` significantly below `O`)*

`Δ_matchup` is significantly **negative** (CI upper bound < 0): `OM` has higher outcome loss than
`O`. Because `Δ_matchup` has **no `min`-bias** (§6), this is *not* a statistical artifact — it is a
genuine out-of-sample **fragmentation cost**: the ~21-column matchup block adds estimation variance
without adding signal, so a strong model generalizes *worse* with it (`THEORY.md` §6.3). This is
**observed on both synthetic worlds** (−0.0078 null, −0.0129 positive), where no matchup effect
exists by construction — so it is a *live* real-data possibility, pre-written here rather than
discovered later. The correct reading is **"no matchup outcome signal plus a fragmentation cost,"**
confirmed with the **support-diagnostics checklist**: (i) are the OM features mostly missing / low-
support (within-game rematches are rare early; batter-vs-family rolling is sparse for low-history
batters)? (ii) does the `first_pitch` slice — where OM's *only* information is cross-PA — show the
same or worse loss? (iii) does the decomposed-vs-direct disagreement flag `OM` (it does on the demo,
§5)? If all three point to fragmentation, `M−` is a statement about *estimation under sparse
features*, not about baseball: matchup memory might still matter, but carrying it as tree features
costs more than it returns here, and the honest move is to route cross-PA memory to WS6's
pooled/embedding representation rather than force it into a tree.

### Cross-reading the grid

The two axes are independent, so the honest headline is a **pair**. The study's *most anticipated*
cell is **H2 × M−** (or **H3 × M−**): within-PA history carries a little outcome value but its fine
order does not, and the matchup block costs out-of-sample — a modest, defensible finding-#2 result
with a clean methodological note on why more features are not more information. The *strongest* cell
is **H1 × M+**: genuine ordered dependence *and* real matchup adaptation, a live signal for the
entire prescriptive phase. The *cleanest null* is **H3 × M0**. Whichever pair fires, it is read under
the firewall: WS3 measures *prediction* (finding #2); only WS4/WS5/WS7 can test whether any of it is
*prescriptive* (finding #3), and only within support.

## 10. Discussion and limitations

**What a GBDT ablation can and cannot conclude.** WS3 measures whether ordered/matchup history
*predicts* outcomes out-of-sample (finding #2). It does **not** establish that *changing* the
sequence would change outcomes (finding #3). The q̂ grid is `E_model[R | s, A=f]`, a conditional
expectation, **not** `E[R | s, do(A=f)]`, a causal effect; identifying the two requires the
ignorability and overlap that public data cannot satisfy (SPEC §0: we never see the pitch that
wasn't thrown; scouting/target/intent are unobserved). This is the finding-#2 / finding-#3 firewall,
and it is exactly what the OPE/RL workstreams *test* rather than assume (`THEORY.md` §8).

**Family granularity and classifier-output labels.** The action and conditioning are at *family*
resolution; within-family velocity/location variation enters as *features* (the D11 physics columns)
but not as *actions*, so a mechanism below the family level is only partially represented. And
`pitch_type` is a Statcast **classifier output**, not the battery's intent — a mislabeled pitch is a
mislabeled action *and* a mislabeled outcome condition; family (8 classes) blunts this but cannot
remove it. The outcome labels are the two-level event tree (decision D5), the minimal tree covering
SPEC §8.2.

**What "GBDT couldn't find it" means (the D34 budget).** A null `Δ_order` under H2/H3 means a strong
tabular model *within a predeclared, equal-across-views 12-combo budget* did not resolve ordered
outcome value — not that no model could. The budget is fixed for fairness (so a loss gap is
information, not search); a much larger search or a different model family (WS6's learned
representation) could in principle differ, which is why the ladder continues past WS3.

**The matchup fragmentation caveat.** As the `M−` branch and the synthetic worlds show, the `OM`
feature block can *cost* out-of-sample where matchup signal is thin. WS3 reports this honestly rather
than hiding it, but it means WS3's matchup read is a *feature-based* one; a pooled/embedding model
(WS6) may resolve cross-PA memory that trees fragment. No catcher/umpire effects (absent from base
Statcast) and a single-metric reward (`−delta_run_exp`) are further scope limits.

**The inheritance map — what WS3 hands downstream.** The propensities `μ(a|s)` → WS4's bandit
(behavior anchoring) and WS7's OPE (importance weights). The q̂ grid and the count-conditional node
values → WS5's MDP (setup-pitch valuation) and WS7's FQE (sequential value). The representation
question WS3 leaves open — can a *learned* encoder see continuous ordered geometry a fixed-slot tree
cannot — → WS6. Every one of them imports WS3's saved artifacts through `load_ws3_artifacts` and
never refits its own outcome or behavior model (decision D33). That is the operational sense in which
WS3 is the centerpiece: it is the load-bearing supply for the entire prescriptive phase.

## 11. Reproducibility appendix

**Phase-2 commands (RUNBOOK WS3.1–3.3).** WS3 is the heaviest CPU step and is split into stages with
per-view checkpointing (a finished `(view, stage)` writes a `.done` marker; a re-run skips it):

```powershell
conda activate statcast; cd ~\pitch-sequencing-research
# WS3.1 behavior mu(a|s) per view
python workstreams/ws3_gbdt_stack/run_ws3.py --table data/processed/decision_table.parquet --out results/ws3/ --stage behavior --views C U L1 O OM --tune --threads 4
# WS3.2 decomposed outcome stack per view
python workstreams/ws3_gbdt_stack/run_ws3.py --table data/processed/decision_table.parquet --out results/ws3/ --stage outcome --views C U L1 O OM --tune --threads 4
# WS3.3 assemble the q-grid + propensities, then score the central table
python workstreams/ws3_gbdt_stack/run_ws3.py --table data/processed/decision_table.parquet --out results/ws3/ --stage assemble --views C U L1 O OM --threads 4
python workstreams/ws3_gbdt_stack/run_ws3.py --table data/processed/decision_table.parquet --out results/ws3/ --stage eval --views C U L1 O OM
```

The synthetic Phase-1 CI equivalents (no data; run the whole pipeline + falsification):
`--synth null --stage all` and `--synth positive --stage all`.

**Artifact inventory (per view, written under `results/ws3/`).**

- `behavior_<view>.joblib` (+ `.json` sidecar: chosen params, top gain features) — the fitted
  `μ(a|s)`.
- `outcome_<view>.joblib` (+ `.json` sidecar: node-value lookups, params, decomposed-vs-direct
  disagreement, top gain features) — the fitted decomposed stack.
- `pred_behavior_<world>_<view>.parquet`, `pred_outcome_<world>_<view>.parquet` — standard-schema
  predictions (`action_probs`; `outcome1`/`outcome2` probs + `exp_reward`/`exp_reward_sd`).
- `qgrid_<world>_<view>.parquet`, `propensity_<world>_<view>.parquet` — the D33 artifacts WS4/5/7
  consume via `load_ws3_artifacts`.
- `ws3_report_<world>.json`, `ws3_<world>.runmeta.json` — the full report + timing/peak-RAM for the
  SPEC §7 Pareto plot.

In [ ]:
import scipy, lightgbm, joblib, sklearn
print("python     :", platform.python_version())
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("scipy      :", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("lightgbm   :", lightgbm.__version__)
print("joblib     :", joblib.__version__)
print("matplotlib :", matplotlib.__version__)
print("seed       :", SEED)
print("data mode  :", DATA_MODE)
print("views      :", VIEWS)
print("N_GAMES    :", N_GAMES, " NB_PARAMS:", NB_PARAMS)